# AgriGuard — Cost-Aware Multi-Agent Crop Monitoring & Response System



**Track:** Agents for Good (Agriculture) — Kaggle "AI Agents: Intensive Vibe Coding Capstone Project"



Small/medium farms lack affordable, autonomous systems to monitor crop health and respond to threats (water stress, pests, adverse weather) in real time. Cloud-only AI solutions are too expensive to run continuously on every sensor reading. **AgriGuard** solves this with a **local, free model (Ollama) for routine decisions**, escalating to an **expensive cloud model (Gemini) only for ambiguous or high-stakes decisions**, with a human (the farm owner) as the final safety net when models disagree or are uncertain.



**Core value proposition:** real-time, low-cost, safe autonomous farm management — with quantifiable cost savings vs. an all-cloud approach.



> **Simulated environment, real decision logic:** this submission does not have physical sensors/drones installed. Sensor readings are generated synthetically (scripted scenarios covering every code path) and actuator actions (irrigation valve, pesticide drone) are simulated — logged, not physically executed. The agent logic, decision architecture, and cost-tracking are fully real; only the physical I/O layer is mocked, and is designed to be swapped for real sensor/drone APIs later (see the Future Work section at the end).

## Architecture



```

Sensors (soil/pest/weather/power) --> Sensor Aggregator Agent (plain Python, no LLM)

         --> ONE consolidated snapshot per decision cycle

         --> Local Decision Agent (ADK LlmAgent + LiteLlm over Ollama, gemma4:e4b)

               |-- confident, routine, not high-stakes --> Safety Rule Layer --> Actuator

               '-- low-confidence / uncertain / high-stakes (spray_pesticide)

                     --> Cloud Reasoning Agent (ADK LlmAgent, Gemini) for a second opinion

                           --> Consensus Agent (deterministic Python)

                                 |-- agree    --> Safety Rule Layer --> Actuator

                                 '-- disagree --> Notification Agent --> farm owner (no action)

```



**Fan-in pattern (Section 3a of the project plan):** sensors never call an LLM directly. Each sensor writes its latest reading to shared aggregator state; the aggregator emits **one** consolidated snapshot per decision cycle. This avoids redundant model calls and race conditions (e.g. a spray decision made from pest data alone, unaware wind just spiked).



**Safety Rule Layer** is deterministic Python — hard-coded limits (wind speed, power, rain forecast) that no LLM decision can override, regardless of model consensus.



**Required competition concepts this satisfies:**

1. **Multi-agent system built on Google ADK** — real `google-adk` (`LlmAgent`, `LiteLlm`, `InMemoryRunner`), not a hand-rolled orchestrator mirroring the concepts.

2. **Security/safety feature** — human-in-the-loop escalation on disagreement/uncertainty, plus a deterministic safety layer that overrides both models.

3. **Cost optimization / model routing** — local-first (free), Gemini only on demand, with quantified cost savings vs. an all-cloud baseline (see the Cost Savings section below).

## Setup — Ollama runs self-contained inside this Kaggle kernel



Per the project plan (Section 3b-i), this notebook must be re-runnable end-to-end by judges without depending on the author's local GPU workstation. Ollama is installed and served **inside the Kaggle kernel itself**, using Kaggle's free GPU quota (T4, 16GB).



**Before running:** confirm this notebook's *Settings → Internet* is **On** (required for `ollama pull` and the Gemini API escalation calls).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

!nohup ollama serve > /kaggle/working/ollama.log 2>&1 &

import time; time.sleep(5)

!curl -s http://localhost:11434/api/version

**Idempotent model pull — never blind-pull.** `ollama pull` re-verifies/re-downloads layers even if the model is already present, wasting minutes on every re-run. The model is ~9.6GB (confirmed via `ollama show`, larger than first estimated) — check `ollama list` first.

In [ ]:
import subprocess



MODEL = "gemma4:e4b"

installed = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout

if MODEL not in installed:

    print(f"Pulling {MODEL} (~9.6GB, first run only)...")

    subprocess.run(["ollama", "pull", MODEL], check=True)

else:

    print(f"{MODEL} already present — skipping pull")

### Install Python dependencies

In [ ]:
%%writefile requirements.txt
google-adk
litellm
python-dotenv
pydantic
google-genai
requests
pytest


In [ ]:
!pip install -q -r requirements.txt

### Configure API keys



Set `GEMINI_API_KEY` below (use Kaggle *Add-ons → Secrets* to avoid hardcoding it — `from kaggle_secrets import UserSecretsClient`). **If left blank, the Cloud Reasoning Agent fails safe**: escalations default to `uncertain` and route to human notification instead of an unverified autonomous action (Section 12.B of the plan) — the Must-have local-only scenarios (1, 2, 5, 6) work with no key at all.

In [ ]:
import os



# os.environ["GEMINI_API_KEY"] = "..."        # or via Kaggle Secrets, see markdown above

os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")

os.environ.setdefault("OLLAMA_MODEL", "gemma4:e4b")

print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))

## Source — same code as local development



The cells below materialize the project's `src/`, `dashboard/`, and `data/` packages into this kernel's working directory via `%%writefile`, so this notebook runs the **identical, already-verified** agent code — not a reimplementation.

In [ ]:
import os
os.makedirs('dashboard', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)
os.makedirs('src/agents', exist_ok=True)
os.makedirs('src/sensors', exist_ok=True)

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
import os

from dotenv import load_dotenv

load_dotenv()

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-2.5-flash"

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "")

# Decision cycle
DECISION_CYCLE_SECONDS = 60
ZONE_IDS = ["zone_1"]

# Escalation
CONFIDENCE_THRESHOLD = 0.85

# Safety Rule Layer (Section 5, step 8 of the plan) — deterministic, model-independent
MAX_WIND_SPEED_KMH_FOR_SPRAY = 15
RAIN_FORECAST_BLOCKS_IRRIGATION = True

# Estimated cost per Gemini call, for the cost-savings dashboard (Section 9)
ESTIMATED_GEMINI_COST_PER_CALL_USD = 0.01

DECISIONS_LOG_PATH = "data/logs/decisions_log.csv"


In [ ]:
%%writefile src/sensors/__init__.py


In [ ]:
%%writefile src/sensors/sensor_simulator.py
import random
from datetime import datetime, timezone


def _now() -> str:
    return datetime.now(timezone.utc).isoformat()


def generate_reading(zone_id: str, overrides: dict | None = None) -> dict:
    """Generate one synthetic sensor reading for a zone.

    Each field simulates an independent sensor writing at its own cadence —
    callers normally set only the fields relevant to a scenario via `overrides`
    and let the rest default to a "healthy baseline" random reading.
    """
    reading = {
        "timestamp": _now(),
        "zone_id": zone_id,
        "soil_moisture_pct": round(random.uniform(60, 75), 1),
        "leaf_health_score": round(random.uniform(0.8, 1.0), 2),
        "pest_risk_score": round(random.uniform(0.0, 0.2), 2),
        "temperature_c": round(random.uniform(22, 30), 1),
        "wind_speed_kmh": round(random.uniform(2, 10), 1),
        "wind_direction": random.choice(["N", "NE", "E", "SE", "S", "SW", "W", "NW"]),
        "power_available": True,
        "forecast_rain_next_6h": False,
    }
    if overrides:
        reading.update(overrides)
    return reading


In [ ]:
%%writefile src/sensors/aggregator_agent.py
from datetime import datetime, timezone

STALE_AFTER_MINUTES = 15


class SensorAggregator:
    """Holds the single source of truth per zone: latest-value-wins per field,
    each timestamped independently (Section 3a of the plan — fan-in pattern).
    No LLM calls happen here.
    """

    def __init__(self):
        self._state: dict[str, dict] = {}

    def ingest(self, reading: dict) -> None:
        zone_id = reading["zone_id"]
        zone_state = self._state.setdefault(zone_id, {})
        for field, value in reading.items():
            if field == "zone_id":
                continue
            zone_state[field] = value

    def snapshot(self, zone_id: str) -> dict:
        """Return one consolidated snapshot for a zone, marking any field whose
        timestamp is older than STALE_AFTER_MINUTES as unknown rather than
        silently reusing a stale cached value (Section 12.D)."""
        zone_state = self._state.get(zone_id, {})
        result = {"zone_id": zone_id, **zone_state}

        timestamp = zone_state.get("timestamp")
        if timestamp:
            age_minutes = (
                datetime.now(timezone.utc) - datetime.fromisoformat(timestamp)
            ).total_seconds() / 60
            if age_minutes > STALE_AFTER_MINUTES:
                result["stale"] = True

        return result


In [ ]:
%%writefile src/agents/__init__.py


In [ ]:
%%writefile src/agents/local_decision_agent.py
import json
from typing import Literal

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types
from pydantic import BaseModel, Field

from src import config

DECISION_PROMPT = """You are the Local Decision Agent for AgriGuard, a crop monitoring system.
Given a single consolidated sensor snapshot for one farm zone, recommend the next
agronomic action based only on plant/soil/pest need: soil_moisture_pct, pest_risk_score,
leaf_health_score, temperature_c.

wind_speed_kmh and power_available are NOT yours to judge — a separate deterministic
safety system enforces those operational constraints after your recommendation. Do not
let them change your action or confidence; recommend the action a plant/pest situation
alone calls for, and let the downstream safety layer block it if it's operationally unsafe.

If soil moisture and pest risk both call for action, pick whichever is more severe
relative to its own healthy range; don't default to "uncertain" just because two
concerns are present.

Snapshot:
{snapshot_json}
"""


class Decision(BaseModel):
    action: Literal["irrigate", "spray_pesticide", "no_action", "uncertain"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str


# gemma4:e4b is a thinking-capable model. Without an explicit output_schema,
# Ollama has no reason to separate its chain-of-thought from the final
# message, so `content` ends up being the raw reasoning instead of JSON
# (confirmed against a live Ollama instance). output_schema forces Ollama's
# native `format` constraint; reasoning_effort="none" (-> LiteLLM's Ollama
# `think` param) turns thinking off, which also cuts per-decision latency
# from ~15s to ~2.5s — this agent needs to be fast and always-on (Section 3b).
_agent = Agent(
    name="local_decision_agent",
    model=LiteLlm(model=f"ollama_chat/{config.OLLAMA_MODEL}", reasoning_effort="none"),
    instruction="You are a careful agronomy decision assistant for a real farm. Be decisive.",
    output_schema=Decision,
    # temperature=0 for reproducible decisions across scripted-scenario demo runs
    # (Section 7 requires the same 6 scenarios to reliably hit the same code paths).
    generate_content_config=types.GenerateContentConfig(temperature=0.0),
)
_runner = InMemoryRunner(agent=_agent, app_name="agriguard")
_USER_ID = "orchestrator"


def decide(snapshot: dict) -> dict:
    """Ask the local model for a first-pass decision on a consolidated snapshot.

    Falls back to "uncertain" (rather than raising) on non-JSON output, so a
    single malformed response escalates to the Cloud Reasoning Agent instead
    of crashing the decision cycle.
    """
    prompt = DECISION_PROMPT.format(snapshot_json=json.dumps(snapshot))
    session = _runner.session_service.create_session_sync(
        app_name="agriguard", user_id=_USER_ID
    )

    reply_text = ""
    for event in _runner.run(
        user_id=_USER_ID,
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
    ):
        if event.is_final_response() and event.content and event.content.parts:
            reply_text = event.content.parts[0].text or ""

    try:
        return json.loads(reply_text.strip().strip("`"))
    except (json.JSONDecodeError, TypeError):
        return {
            "action": "uncertain",
            "confidence": 0.0,
            "reasoning": f"local model returned non-JSON output: {reply_text!r}",
        }


In [ ]:
%%writefile src/agents/cloud_reasoning_agent.py
import json
from typing import Literal

from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types
from pydantic import BaseModel, Field

from src import config

DECISION_PROMPT = """You are the Cloud Reasoning Agent for AgriGuard, a crop monitoring system.
The Local Decision Agent already looked at this snapshot and was uncertain or its
proposed action was high-stakes enough to need a second opinion. Give your own
independent read.

Recommend the next agronomic action based only on plant/soil/pest need:
soil_moisture_pct, pest_risk_score, leaf_health_score, temperature_c.

wind_speed_kmh and power_available are NOT yours to judge — a separate deterministic
safety system enforces those operational constraints after your recommendation. Do not
let them change your action or confidence.

If soil moisture and pest risk both call for action, pick whichever is more severe
relative to its own healthy range; don't default to "uncertain" just because two
concerns are present.

Snapshot:
{snapshot_json}

Local Decision Agent's opinion (for context only — form your own independent judgment,
don't just agree with it):
{local_decision_json}
"""


class Decision(BaseModel):
    action: Literal["irrigate", "spray_pesticide", "no_action", "uncertain"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str


_agent = Agent(
    name="cloud_reasoning_agent",
    model=config.GEMINI_MODEL,
    instruction="You are a careful agronomy decision assistant for a real farm. Be decisive.",
    output_schema=Decision,
    generate_content_config=types.GenerateContentConfig(temperature=0.0),
)
_runner = InMemoryRunner(agent=_agent, app_name="agriguard")
_USER_ID = "orchestrator"


def decide(snapshot: dict, local_decision: dict) -> dict:
    """Ask Gemini for a second opinion on a snapshot the local model flagged.

    Fails safe (Section 12.B): on any API error, timeout, or non-JSON output,
    returns "uncertain" rather than raising, so the caller falls through to
    the human-escalation path instead of taking an unverified autonomous action.
    """
    if not config.GEMINI_API_KEY:
        return {
            "action": "uncertain",
            "confidence": 0.0,
            "reasoning": "GEMINI_API_KEY not configured — cannot reach Cloud Reasoning Agent",
        }

    prompt = DECISION_PROMPT.format(
        snapshot_json=json.dumps(snapshot),
        local_decision_json=json.dumps(local_decision),
    )

    try:
        session = _runner.session_service.create_session_sync(
            app_name="agriguard", user_id=_USER_ID
        )

        reply_text = ""
        for event in _runner.run(
            user_id=_USER_ID,
            session_id=session.id,
            new_message=types.Content(role="user", parts=[types.Part(text=prompt)]),
        ):
            if event.is_final_response() and event.content and event.content.parts:
                reply_text = event.content.parts[0].text or ""

        return json.loads(reply_text.strip().strip("`"))
    except Exception as exc:
        return {
            "action": "uncertain",
            "confidence": 0.0,
            "reasoning": f"Cloud Reasoning Agent call failed: {exc!r}",
        }


In [ ]:
%%writefile src/agents/consensus_agent.py
def decide(local_decision: dict, cloud_decision: dict) -> tuple[str, str]:
    """Deterministic agreement check between the Local and Cloud decisions
    (Section 5, step 7; Section 8 — no LLM involved in this comparison).

    Returns (resolution, decision_source):
      - resolution is "agree" or "disagree"
      - decision_source is one of metrics.record's decision_source values:
        "escalated_to_gemini" on agreement, "escalated_to_human" on disagreement.
    """
    local_action = local_decision.get("action", "uncertain")
    cloud_action = cloud_decision.get("action", "uncertain")

    if local_action == cloud_action and local_action != "uncertain":
        return "agree", "escalated_to_gemini"

    return "disagree", "escalated_to_human"


In [ ]:
%%writefile src/agents/safety_rules.py
from src import config


def apply(snapshot: dict, proposed_action: str) -> tuple[str, str | None]:
    """Deterministic, model-independent safety gate (Section 5, step 8).

    Returns (final_action, block_reason). block_reason is None if the
    proposed_action was allowed through unchanged.
    """
    if proposed_action == "spray_pesticide":
        if snapshot.get("wind_speed_kmh", 0) > config.MAX_WIND_SPEED_KMH_FOR_SPRAY:
            return "no_action", "blocked: wind_speed exceeds spray safety limit"
        if not snapshot.get("power_available", True):
            return "no_action", "blocked: no power available for drone operation"

    if proposed_action == "irrigate":
        if config.RAIN_FORECAST_BLOCKS_IRRIGATION and snapshot.get(
            "forecast_rain_next_6h", False
        ):
            return "no_action", "blocked: rain forecast within next 6 hours"

    return proposed_action, None


In [ ]:
%%writefile src/agents/actuator_agent.py
from datetime import datetime, timezone


def execute(zone_id: str, action: str, block_reason: str | None) -> dict:
    """Simulated actuator — logs what would have happened rather than driving
    real hardware (Section 2 of the plan: physical I/O is explicitly mocked)."""
    result = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "zone_id": zone_id,
        "action": action,
        "executed": action != "no_action",
        "block_reason": block_reason,
    }
    print(f"[ACTUATOR] zone={zone_id} action={action} executed={result['executed']}"
          + (f" ({block_reason})" if block_reason else ""))
    return result


In [ ]:
%%writefile src/agents/notification_agent.py
import requests

from src import config


def notify(zone_id: str, message: str) -> None:
    """Alerts the farm owner (Section 3c / Section 5 step 7).

    Uses Telegram's Bot API if TELEGRAM_BOT_TOKEN/TELEGRAM_CHAT_ID are set,
    else falls back to console/log — Telegram is not a hard dependency for
    this tier (Should-have, Section 10).
    """
    full_message = f"[AgriGuard] zone={zone_id}: {message}"

    if config.TELEGRAM_BOT_TOKEN and config.TELEGRAM_CHAT_ID:
        url = f"https://api.telegram.org/bot{config.TELEGRAM_BOT_TOKEN}/sendMessage"
        try:
            requests.post(
                url,
                json={"chat_id": config.TELEGRAM_CHAT_ID, "text": full_message},
                timeout=10,
            )
            return
        except requests.RequestException as exc:
            print(f"[NOTIFICATION] Telegram send failed ({exc!r}), falling back to console")

    print(f"[NOTIFICATION] {full_message}")


In [ ]:
%%writefile src/orchestrator.py
import json

from src import config, metrics
from src.agents import (
    actuator_agent,
    cloud_reasoning_agent,
    consensus_agent,
    local_decision_agent,
    notification_agent,
    safety_rules,
)
from src.sensors.aggregator_agent import SensorAggregator
from src.sensors.sensor_simulator import generate_reading


HIGH_STAKES_ACTIONS = {"spray_pesticide"}


def run_cycle(aggregator: SensorAggregator, zone_id: str) -> dict:
    """One decision cycle for a zone: aggregator snapshot -> local decision
    agent -> (if confident and routine) safety gate -> actuator, OR (if low
    confidence, uncertain, or high-stakes per Section 5 step 5) cloud
    escalation -> consensus -> safety gate -> actuator / human notification.
    """
    snapshot = aggregator.snapshot(zone_id)
    decision = local_decision_agent.decide(snapshot)

    action = decision.get("action", "uncertain")
    confidence = decision.get("confidence", 0.0)

    needs_escalation = (
        confidence < config.CONFIDENCE_THRESHOLD
        or action == "uncertain"
        or action in HIGH_STAKES_ACTIONS
    )

    if not needs_escalation:
        final_action, block_reason = safety_rules.apply(snapshot, action)
        result = actuator_agent.execute(zone_id, final_action, block_reason)
        metrics.record(zone_id, "local", final_action, confidence, block_reason)
        return result

    print(f"[ORCHESTRATOR] zone={zone_id} escalating to Cloud Reasoning Agent "
          f"— local decision: {decision}")
    cloud_decision = cloud_reasoning_agent.decide(snapshot, decision)
    resolution, decision_source = consensus_agent.decide(decision, cloud_decision)

    if resolution == "agree":
        agreed_action = cloud_decision["action"]
        agreed_confidence = min(confidence, cloud_decision.get("confidence", 0.0))
        final_action, block_reason = safety_rules.apply(snapshot, agreed_action)
        result = actuator_agent.execute(zone_id, final_action, block_reason)
        metrics.record(zone_id, decision_source, final_action, agreed_confidence, block_reason)
        return result

    notification_agent.notify(
        zone_id,
        f"models disagree, no action taken — local: {decision}, cloud: {cloud_decision}",
    )
    metrics.record(zone_id, decision_source, "no_action", confidence, "disagreement — human escalation")
    return {"zone_id": zone_id, "action": "no_action", "executed": False, "block_reason": "disagreement — human escalation"}


def run_scripted_scenarios(scenarios_path: str = "data/scenarios.json"):
    """Feeds each scripted scenario (Section 7) through one decision cycle,
    for a reliable, reproducible demo instead of live randomness."""
    with open(scenarios_path) as f:
        scenarios = json.load(f)

    aggregator = SensorAggregator()
    for scenario in scenarios:
        print(f"\n=== Scenario {scenario['id']}: {scenario['name']} ===")
        reading = generate_reading(scenario["zone_id"], overrides=scenario["overrides"])
        aggregator.ingest(reading)
        run_cycle(aggregator, scenario["zone_id"])

    print("\n=== Cost summary ===")
    print(metrics.summary())


if __name__ == "__main__":
    run_scripted_scenarios()


In [ ]:
%%writefile src/metrics.py
import csv
import os
from datetime import datetime, timezone

from src import config

_FIELDNAMES = [
    "timestamp",
    "zone_id",
    "decision_source",
    "action",
    "confidence",
    "block_reason",
    "estimated_cost_usd",
]


def _ensure_log_file():
    os.makedirs(os.path.dirname(config.DECISIONS_LOG_PATH), exist_ok=True)
    if not os.path.exists(config.DECISIONS_LOG_PATH):
        with open(config.DECISIONS_LOG_PATH, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=_FIELDNAMES).writeheader()


def record(zone_id: str, decision_source: str, action: str, confidence: float, block_reason: str | None):
    """decision_source is one of: 'local', 'escalated_to_gemini', 'escalated_to_human'."""
    _ensure_log_file()
    estimated_cost = (
        config.ESTIMATED_GEMINI_COST_PER_CALL_USD if decision_source != "local" else 0.0
    )
    with open(config.DECISIONS_LOG_PATH, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=_FIELDNAMES).writerow(
            {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "zone_id": zone_id,
                "decision_source": decision_source,
                "action": action,
                "confidence": confidence,
                "block_reason": block_reason or "",
                "estimated_cost_usd": estimated_cost,
            }
        )


def summary() -> dict:
    """Cost-savings story (Section 9): % resolved locally vs escalated, and
    actual cost vs. a hypothetical all-cloud baseline."""
    _ensure_log_file()
    with open(config.DECISIONS_LOG_PATH, newline="") as f:
        rows = list(csv.DictReader(f))

    total = len(rows)
    if total == 0:
        return {"total_decisions": 0}

    local_count = sum(1 for r in rows if r["decision_source"] == "local")
    gemini_count = sum(1 for r in rows if r["decision_source"] == "escalated_to_gemini")
    human_count = sum(1 for r in rows if r["decision_source"] == "escalated_to_human")
    actual_cost = sum(float(r["estimated_cost_usd"]) for r in rows)
    hypothetical_all_cloud_cost = total * config.ESTIMATED_GEMINI_COST_PER_CALL_USD

    return {
        "total_decisions": total,
        "pct_local": round(100 * local_count / total, 1),
        "pct_escalated_to_gemini": round(100 * gemini_count / total, 1),
        "pct_escalated_to_human": round(100 * human_count / total, 1),
        "actual_cost_usd": round(actual_cost, 4),
        "hypothetical_all_cloud_cost_usd": round(hypothetical_all_cloud_cost, 4),
        "savings_usd": round(hypothetical_all_cloud_cost - actual_cost, 4),
        "savings_pct": round(
            100 * (hypothetical_all_cloud_cost - actual_cost) / hypothetical_all_cloud_cost, 1
        )
        if hypothetical_all_cloud_cost
        else 0.0,
    }


In [ ]:
%%writefile dashboard/__init__.py


In [ ]:
%%writefile dashboard/dashboard.py
"""Generates a self-contained HTML cost/decision dashboard from
data/logs/decisions_log.csv (Section 9 of the plan — the cost-savings story).

Usage: python -m dashboard.dashboard
Writes dashboard/report.html.
"""

import csv
import html
import os

from src import config, metrics

REPORT_PATH = os.path.join(os.path.dirname(__file__), "report.html")

# Categorical slots (fixed order, from the project's data-viz palette), one
# per decision source — never reassigned based on which sources are present.
SOURCE_COLORS = {
    "local": "#2a78d6",  # slot 1, blue
    "escalated_to_gemini": "#1baf7a",  # slot 2, aqua
    "escalated_to_human": "#eda100",  # slot 3, yellow
}
SOURCE_LABELS = {
    "local": "Local (Ollama, $0)",
    "escalated_to_gemini": "Escalated to Gemini",
    "escalated_to_human": "Escalated to human",
}


def _read_rows() -> list[dict]:
    if not os.path.exists(config.DECISIONS_LOG_PATH):
        return []
    with open(config.DECISIONS_LOG_PATH, newline="") as f:
        return list(csv.DictReader(f))


def _bar(label: str, count: int, total: int, color: str) -> str:
    pct = round(100 * count / total, 1) if total else 0.0
    return f"""
    <div class="bar-row">
      <div class="bar-label">{html.escape(label)}</div>
      <div class="bar-track">
        <div class="bar-fill" style="width:{pct}%; background:{color};"></div>
      </div>
      <div class="bar-value">{count} ({pct}%)</div>
    </div>"""


def _cost_bar(label: str, value: float, max_value: float, color: str) -> str:
    pct = round(100 * value / max_value, 1) if max_value else 0.0
    return f"""
    <div class="bar-row">
      <div class="bar-label">{html.escape(label)}</div>
      <div class="bar-track">
        <div class="bar-fill" style="width:{pct}%; background:{color};"></div>
      </div>
      <div class="bar-value">${value:.4f}</div>
    </div>"""


def _table(rows: list[dict]) -> str:
    if not rows:
        return "<p class='muted'>No decisions logged yet.</p>"
    body = "\n".join(
        f"<tr><td>{html.escape(r['timestamp'])}</td><td>{html.escape(r['zone_id'])}</td>"
        f"<td>{html.escape(SOURCE_LABELS.get(r['decision_source'], r['decision_source']))}</td>"
        f"<td>{html.escape(r['action'])}</td><td>{html.escape(r['confidence'])}</td>"
        f"<td>{html.escape(r['block_reason'])}</td><td>${float(r['estimated_cost_usd']):.4f}</td></tr>"
        for r in rows
    )
    return f"""
    <table>
      <thead><tr><th>Timestamp</th><th>Zone</th><th>Source</th><th>Action</th>
      <th>Confidence</th><th>Block reason</th><th>Cost</th></tr></thead>
      <tbody>{body}</tbody>
    </table>"""


def generate() -> str:
    rows = _read_rows()
    summary = metrics.summary()
    total = summary.get("total_decisions", 0)

    source_counts = {source: 0 for source in SOURCE_COLORS}
    for r in rows:
        if r["decision_source"] in source_counts:
            source_counts[r["decision_source"]] += 1

    source_bars = "\n".join(
        _bar(SOURCE_LABELS[source], count, total, SOURCE_COLORS[source])
        for source, count in source_counts.items()
    )

    actual_cost = summary.get("actual_cost_usd", 0.0)
    hypothetical_cost = summary.get("hypothetical_all_cloud_cost_usd", 0.0)
    max_cost = max(actual_cost, hypothetical_cost, 0.0001)
    cost_bars = (
        _cost_bar("Actual (cost-aware cascade)", actual_cost, max_cost, "#2a78d6")
        + _cost_bar("Hypothetical (all-cloud)", hypothetical_cost, max_cost, "#e34948")
    )

    stat_tiles = "".join(
        f"<div class='tile'><div class='tile-value'>{value}</div><div class='tile-label'>{label}</div></div>"
        for label, value in [
            ("Total decisions", total),
            ("Savings vs. all-cloud", f"{summary.get('savings_pct', 0)}%"),
            ("Actual cost", f"${actual_cost:.4f}"),
            ("Hypothetical all-cloud cost", f"${hypothetical_cost:.4f}"),
        ]
    )

    return f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>AgriGuard — Cost &amp; Decision Dashboard</title>
<style>
  :root {{
    --surface-1: #fcfcfb; --text-primary: #0b0b0b; --text-secondary: #52514e;
    --border: #e3e2dc;
  }}
  @media (prefers-color-scheme: dark) {{
    :root {{ --surface-1: #1a1a19; --text-primary: #ffffff; --text-secondary: #c3c2b7; --border: #3a3a37; }}
  }}
  :root[data-theme="dark"] {{ --surface-1: #1a1a19; --text-primary: #ffffff; --text-secondary: #c3c2b7; --border: #3a3a37; }}
  :root[data-theme="light"] {{ --surface-1: #fcfcfb; --text-primary: #0b0b0b; --text-secondary: #52514e; --border: #e3e2dc; }}
  body {{ background: var(--surface-1); color: var(--text-primary); font-family: system-ui, sans-serif; margin: 0; padding: 2rem; }}
  h1 {{ font-size: 1.4rem; margin-bottom: 0.25rem; }}
  .subtitle {{ color: var(--text-secondary); margin-bottom: 2rem; }}
  .tiles {{ display: flex; gap: 1rem; flex-wrap: wrap; margin-bottom: 2rem; }}
  .tile {{ border: 1px solid var(--border); border-radius: 8px; padding: 1rem 1.5rem; min-width: 160px; }}
  .tile-value {{ font-size: 1.6rem; font-weight: 600; }}
  .tile-label {{ color: var(--text-secondary); font-size: 0.85rem; margin-top: 0.25rem; }}
  section {{ margin-bottom: 2rem; }}
  h2 {{ font-size: 1.05rem; margin-bottom: 1rem; }}
  .bar-row {{ display: flex; align-items: center; gap: 0.75rem; margin-bottom: 0.6rem; }}
  .bar-label {{ width: 220px; font-size: 0.9rem; color: var(--text-secondary); flex-shrink: 0; }}
  .bar-track {{ flex: 1; background: var(--border); border-radius: 4px; height: 18px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 4px; }}
  .bar-value {{ width: 110px; text-align: right; font-size: 0.85rem; flex-shrink: 0; }}
  table {{ border-collapse: collapse; width: 100%; font-size: 0.85rem; }}
  th, td {{ border-bottom: 1px solid var(--border); padding: 0.4rem 0.6rem; text-align: left; }}
  th {{ color: var(--text-secondary); font-weight: 500; }}
  .muted {{ color: var(--text-secondary); }}
  .table-wrap {{ overflow-x: auto; }}
</style>
</head>
<body>
  <h1>AgriGuard — Cost &amp; Decision Dashboard</h1>
  <p class="subtitle">Cost-aware multi-agent crop monitoring — decision source breakdown and cascading-cost savings (Section 9)</p>

  <div class="tiles">{stat_tiles}</div>

  <section>
    <h2>Decisions by source</h2>
    {source_bars}
  </section>

  <section>
    <h2>Cost: cascading vs. all-cloud baseline</h2>
    {cost_bars}
  </section>

  <section>
    <h2>Decision log</h2>
    <div class="table-wrap">{_table(rows)}</div>
  </section>
</body>
</html>"""


if __name__ == "__main__":
    report_html = generate()
    with open(REPORT_PATH, "w") as f:
        f.write(report_html)
    print(f"Dashboard written to {REPORT_PATH}")


In [ ]:
%%writefile data/scenarios.json
[
  {
    "id": 1,
    "name": "clear_cut_irrigation_need",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 12,
      "forecast_rain_next_6h": false
    }
  },
  {
    "id": 2,
    "name": "clear_cut_no_action",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 65,
      "pest_risk_score": 0.05,
      "forecast_rain_next_6h": false
    }
  },
  {
    "id": 3,
    "name": "ambiguous_pest_signal",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 65,
      "pest_risk_score": 0.8,
      "leaf_health_score": 0.6,
      "temperature_c": 27
    }
  },
  {
    "id": 4,
    "name": "cloud_escalation_consensus",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 65,
      "pest_risk_score": 0.7,
      "leaf_health_score": 0.4,
      "temperature_c": 30
    }
  },
  {
    "id": 5,
    "name": "safety_override_wind",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 65,
      "pest_risk_score": 0.9,
      "wind_speed_kmh": 28
    }
  },
  {
    "id": 6,
    "name": "power_constraint",
    "zone_id": "zone_1",
    "overrides": {
      "soil_moisture_pct": 65,
      "pest_risk_score": 0.9,
      "power_available": false
    }
  }
]


## Run the scripted-scenario demo



Runs all 6 hand-crafted scenarios (Section 7 of the plan) through one decision cycle each — chosen to exercise every code path: clear-cut irrigation, clear-cut no-action, cloud escalation on a high-stakes pest signal, a second escalation+consensus case, and two deterministic safety-layer overrides (wind, power) that block spraying regardless of model agreement.

In [ ]:
import sys

sys.path.insert(0, ".")



from src.orchestrator import run_scripted_scenarios



run_scripted_scenarios()

## Cost-savings story (Section 9)



The single most memorable number for judges: **actual cascading cost vs. a hypothetical baseline where every decision went straight to Gemini.**

In [ ]:
from src import metrics



summary = metrics.summary()

for k, v in summary.items():

    print(f"{k:35s}: {v}")

### Visual dashboard

In [ ]:
from dashboard.dashboard import generate

from IPython.display import HTML



HTML(generate())

## Required-concepts mapping



| # | Concept | Where in this notebook |

|---|---|---|

| 1 | **Multi-agent system on Google ADK** | `local_decision_agent.py` and `cloud_reasoning_agent.py` are real `google.adk.agents.Agent` instances run through `InMemoryRunner`; the Local agent wraps Ollama via `LiteLlm("ollama_chat/gemma4:e4b")`, the Cloud agent uses ADK's native Gemini support. `consensus_agent.py` and `safety_rules.py` are deliberately **plain deterministic Python**, not agents — safety logic must never be delegated to a model. |

| 2 | **Security / safety feature** | `safety_rules.py` is a hard-coded, model-independent gate (wind speed, power availability, rain forecast) that downgrades any unsafe action to `no_action` regardless of model consensus. `consensus_agent.py` + `notification_agent.py` implement human-in-the-loop escalation: the system never acts autonomously when the two models disagree or are both uncertain. |

| 3 | **Cost optimization / model routing** | `orchestrator.py`'s `run_cycle` resolves routine, confident decisions locally (\$0) and only escalates to Gemini on low confidence, `"uncertain"`, or the high-stakes `spray_pesticide` action — see the cost summary above for the actual observed savings vs. an all-cloud baseline. |

## Simulated vs. real (Section 2)



This is explicitly a simulated environment: sensor readings (`sensor_simulator.py`) are synthetic/scripted, and actuator actions (`actuator_agent.py`) are logged, not physically executed. The agent logic, decision architecture, and cost-tracking are fully real; only the physical I/O layer is mocked. A real deployment would swap `sensor_simulator.py` for actual soil/pest/weather sensor APIs and `actuator_agent.py` for real irrigation-valve / pesticide-drone control — the rest of the pipeline (aggregator, decision cascade, safety layer, consensus, logging) is unchanged.

## Production deployment note



This notebook runs Ollama for fast iteration and judge re-runnability. For a real, unattended field deployment on lower-power edge hardware, the recommended production runtime is a `llama.cpp` server instead: no automatic model unloading (avoiding Ollama's cold-start reload penalty), and finer control over context size, batching, and quantization for a resource-constrained device actually sitting on the farm. `local_decision_agent.py` only depends on `OLLAMA_HOST` being OpenAI/Ollama-API-compatible, so this swap doesn't require touching the agent logic.

## Key bottlenecks identified while building this (Section 12)



- **Rural/farm internet unreliability** — Gemini escalation and notifications fail safe: on any API error/timeout/quota exhaustion, the system defaults to `uncertain` -> human escalation -> `no_action`, never an unverified autonomous action. **Observed directly** during development: the free-tier Gemini quota (20 requests/day for `gemini-2.5-flash`) was exhausted mid-testing, and the fail-safe path activated exactly as designed instead of crashing.

- **Thinking-capable local models leaking chain-of-thought as output** — `gemma4:e4b` is a thinking-capable model; without an explicit `output_schema` (forcing Ollama's structured `format` constraint) its raw reasoning came back as the response instead of JSON. Fixed via ADK's `output_schema` + `reasoning_effort="none"` (disables thinking, ~6x latency reduction per decision).

- **Decision-relevant fields need unambiguous "healthy baseline" ranges** — an earlier random soil-moisture baseline (35-55%) sat in a gray zone the model read as "needs irrigation," making the scripted safety-override scenarios non-deterministic across runs. Fixed by widening the healthy baseline and pinning decision-relevant fields explicitly in each scenario's overrides.

- **Genuine model disagreement is empirically rare** between two well-calibrated models (`gemma4:e4b` and `gemini-2.5-flash`) given the same structured, explicit sensor data — they converge on the same action far more often than they diverge. The disagreement / human-escalation code path is directly unit-tested (`tests/test_consensus_agent.py`) rather than relied upon to reproduce live on every demo run.